### Preparing the data
In this notebook, we demonstrate how to prepare the Mouse Smart-seq dataset, which is a single-cell dataset was released as part of a transcriptomic cell types study in [Tasic et al., 2018](https://portal.brain-map.org/atlases-and-data/rnaseq/mouse-v1-and-alm-smart-seq). The dataset includes RNA sequencing of neurons from the anterolateral motor cortex (ALM) and primary visual cortex (VISp) regions of adult mice using Smart-seq (SSv4) platform. 

In [1]:
import os

# TODO: Understand
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
assert os.path.basename(os.getcwd()) == "distributed-vae"

In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial

import toml
import pandas as pd
import numpy as np
import anndata as ad
import sklearn.preprocessing as skp
from scipy.sparse import csr_matrix

from mmidas.utils.taxonomy import HTree
from mmidas.utils.tools import (
    logcpm,
    logcpm2,
    get_paths,
    join_path,
    path_exists,
    join2,
    l1,
    normalize,
    indices_of,
    index_of,
    join_data,
)

%matplotlib inline

import warnings

warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Download ```zip``` files and locate them within the ```data``` folder. 

In [3]:
CONFIG_PATH = "config.toml"
DATASET = "mouse_smartseq"
DATA_PATH = "data"
NEURONAL = {"GABAergic", "Glutamatergic"}

config = toml.load(CONFIG_PATH)

In [4]:
# Values are transcript counts of genes

# Load the mouse Smart-seq VISp data
exon_vis_df = pd.read_csv(
    join_data("mouse_VISp_2018-06-14_exon-matrix.csv")
)  # TODO: understand
ann_vis_df = pd.read_csv(
    join_data("mouse_VISp_2018-06-14_samples-columns.csv"), encoding="unicode_escape"
)  # primary visual cortex

# Load the mouse Smart-seq ALM data
exon_alm_df = pd.read_csv(join_data("mouse_ALM_2018-06-14_exon-matrix.csv"))
ann_alm_df = pd.read_csv(
    join_data("mouse_ALM_2018-06-14_samples-columns.csv"), encoding="unicode_escape"
)  # anterior lateral motor cortex

print(f"Total number of cells in VISp and ALM: {len(ann_vis_df)}, {len(ann_alm_df)}")

Total number of cells in VISp and ALM: 15413, 10068


In [5]:
# Numpy arrays of the gene expression data (gene counts per cell)
xss_vis = exon_vis_df.values[:, 1:].T
xss_alm = exon_alm_df.values[:, 1:].T

assert len(xss_vis) == len(ann_vis_df)
assert len(xss_alm) == len(ann_alm_df)

In [ ]:
# Get the neuronal cells across brain regions

is_neuron_vis = ann_vis_df["class"].isin(NEURONAL)
is_neuron_alm = ann_alm_df["class"].isin(NEURONAL)

xss_vis_neuron = xss_vis[is_neuron_vis]
xss_alm_neuron = xss_alm[is_neuron_alm]

ann = pd.concat(
    [ann_vis_df[is_neuron_vis], ann_alm_df[is_neuron_alm]], ignore_index=True
)
xss = np.concatenate([xss_vis_neuron, xss_alm_neuron])

# Normalized counts values using LogCPM
xss = logcpm2(xss)
print(np.sum(xss, axis=1))

[30890.15859407 34090.13980254 35085.63428565 ... 34077.15380524
 31090.81791427 35629.482184  ]


In [39]:
genes_df = pd.read_csv(
    join_data("mouse_ALM_2018-06-14_genes-rows.csv")
)  # list of all genes in the dataset
genes_slc_df = pd.read_csv(
    join_data(config[DATASET]["ref_gene_file"])
)  # selected genes for mouse Smart-seq data analysis

print(genes_df[41530:41540])
print("-" * 100)
print(
    f"Total number of genes: {len(genes_df)}, Number of selected genes: {len(genes_slc_df)}"
)

      gene_symbol    gene_id chromosome  gene_entrez_id  \
41530      Sssca1  500741647         19           56390   
41531         Sst  500737291         16           20604   
41532       Sstr1  500729687         12           20605   
41533       Sstr2  500728684         11           20606   
41534       Sstr3  500736064         15           20607   
41535       Sstr4  500704969          2           20608   
41536       Sstr5  500738797         17           20609   
41537       Ssty1  500745186          Y           20611   
41538       Ssty2  500745340          Y           70009   
41539        Ssu2  500714992          6          243612   

                                               gene_name  
41530  Sjogren''s syndrome/scleroderma autoantigen 1 ...  
41531                                       somatostatin  
41532                            somatostatin receptor 1  
41533                            somatostatin receptor 2  
41534                            somatostatin receptor 

Filter out genes that were not selected, as well as two categories of cells: low quality cells, and those belonging to ```CR``` and ```Meis2``` subclasses.

In [38]:
genes = genes_slc_df.genes.values
ixs = indices_of(genes_df["gene_symbol"].values, genes)
xss = xss[:, ixs]

# remove low quality cells and CR and Meis2 subclasses
mask = (
    (ann["cluster"] != "Low Quality")
    & (ann["cluster"] != "CR Lhx5")
    & (ann["cluster"] != "Meis2 Adamts19")
)
df_anno = ann[mask].reset_index()
xss = xss[mask, :]

print(f"final shape of normalized gene expresion matix: {xss.shape}")

final shape of normalized gene expresion matix: (22365, 5032)


Build an AnnData object for the dataloader. 

In [49]:
# load the tree.csv to obtain colors for t-types on the taxonomies
tree = HTree(htree_file=join_data(config[DATASET]["htree_file"]))
ttypes = tree.child[tree.isleaf]
colors = tree.col[tree.isleaf]
df_anno.rename(columns={"seq_name": "sample_id", "class": "class_label"})

# rename two cell types according to the taxonomy
df_anno["cluster"][df_anno["cluster"] == "L6b VISp Col8a1 Rprm"] = "L6b Col8a1 Rprm"
df_anno["cluster"][df_anno["cluster"] == "L6 CT ALM Nxph2 Sla"] = "L6 CT Nxph2 Sla"

In [56]:
# save data as AnnData object
sub_df = df_anno[
    [
        "sample_name",
        "sample_id",
        "seq_batch",
        "sex",
        "brain_hemisphere",
        "brain_region",
        "brain_subregion",
        "class",
        "subclass",
        "cluster",
        "confusion_score",
    ]
]
adata = ad.AnnData(X=csr_matrix(xss), obs=sub_df)
adata.var_names = genes
adata.obs_names = sub_df.sample_id.values
adata.write_h5ad(join_data(config[DATASET]["anndata_file"]))